# Day H6 — 외부 검증과 데이터셋 이동: 다른 기관에 가면 무엇이 무너지나

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/H-clinical-prediction/day-H6.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/H-clinical-prediction/day-H6.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 세 가지 이동이 남기는 흔적

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(8)
def site(n, age_mean, base, gain, noise):
    """기관 하나의 가상 자료. gain=기기 증폭률, noise=기기 잡음, base=기저 위험(logit)."""
    age = rng.normal(age_mean, 8, n)
    sex = rng.integers(0, 2, n)                               # 1 = 여성
    tremor_true = rng.normal(0, 1, n)
    y = rng.random(n) < 1 / (1 + np.exp(-(base + 0.06 * (age - 65) + 1.0 * tremor_true)))
    tremor_meas = gain * tremor_true + rng.normal(0, noise, n)  # 기기가 "보는" 값
    return pd.DataFrame({"age": age, "female": sex, "tremor": tremor_meas, "y": y.astype(int)})

dev  = site(1500, 65, -1.4, gain=1.0, noise=0.3)    # 개발 기관
ext1 = site(1000, 65, -1.4, gain=1.0, noise=0.3)    # 같은 조건의 다른 표본
ext2 = site(1000, 72, -2.2, gain=1.6, noise=0.3)    # 더 고령, 기저 위험 낮음, 기기 증폭률 1.6배
ext3 = site(1000, 65, -1.4, gain=1.0, noise=1.2)    # 기기 잡음이 큼
X = ["age", "female", "tremor"]
m = LogisticRegression().fit(dev[X], dev.y)

def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

def evaluate(name, d, p):
    lp = logit(p)
    slope = sm.Logit(d.y, sm.add_constant(lp)).fit(disp=0).params.iloc[1]
    citl = sm.GLM(d.y, np.ones((len(d), 1)), family=sm.families.Binomial(), offset=lp).fit().params.iloc[0]
    print(f"{name:22s} AUC {roc_auc_score(d.y, p):.3f} | 관찰 {d.y.mean():.3f} 예측 {p.mean():.3f} | 절편 {citl:+.2f} 기울기 {slope:.2f}")

for name, d in [("같은 조건", ext1), ("고령·저위험·증폭 1.6배", ext2), ("잡음 큰 기기", ext3)]:
    evaluate(name, d, m.predict_proba(d[X])[:, 1])

### 예시 2 — 새 기관 200명으로 재보정하기

외부 기관 2의 처음 200명으로 재보정하고, 나머지 800명으로 확인합니다.

In [ ]:
# 예시 2 — 새 기관의 일부(200명)로 재보정하고 나머지(800명)로 확인
local, hold = ext2.iloc[:200], ext2.iloc[200:]
lp_local = logit(m.predict_proba(local[X])[:, 1])
lp_hold = logit(m.predict_proba(hold[X])[:, 1])
a = sm.GLM(local.y, np.ones((200, 1)), family=sm.families.Binomial(), offset=lp_local).fit().params.iloc[0]
ab = sm.Logit(local.y, sm.add_constant(lp_local)).fit(disp=0).params
evaluate("재보정 전", hold, 1 / (1 + np.exp(-lp_hold)))
evaluate("절편만 재보정", hold, 1 / (1 + np.exp(-(a + lp_hold))))
evaluate("절편+기울기 재보정", hold, 1 / (1 + np.exp(-(ab.iloc[0] + ab.iloc[1] * lp_hold))))

### 예시 3 — 하위집단 성능을 구간과 함께

In [ ]:
# 예시 3 — 하위집단 성능: 숫자 하나 대신 집단별 숫자와 불확실성
d = ext1.copy()
d["p"] = m.predict_proba(d[X])[:, 1]
d["age_group"] = np.where(d.age >= 70, "70+", "<70")
def boot_auc(sub, B=1000):
    vals = []
    for _ in range(B):
        i = rng.integers(0, len(sub), len(sub))
        s = sub.iloc[i]
        if s.y.nunique() == 2:
            vals.append(roc_auc_score(s.y, s.p))
    return np.percentile(vals, [2.5, 97.5])
for col in ["female", "age_group"]:
    for g, sub in d.groupby(col):
        lo, hi = boot_auc(sub)
        sens = (sub.p[sub.y == 1] >= 0.25).mean()
        print(f"{col}={g!s:4s} n={len(sub):4d} 사건 {sub.y.sum():3d} | AUC {roc_auc_score(sub.y, sub.p):.3f} ({lo:.3f}–{hi:.3f})"
              f" | 임계값 0.25 민감도 {sens:.2f} | 관찰 {sub.y.mean():.2f} 예측 {sub.p.mean():.2f}")

### 예시 4 — 배포 후 감시: O/E

In [ ]:
# 예시 4 — 배포 후 감시: 달마다 관찰/예측 비(O/E)와 3개월 누적 O/E
log = []
for month in range(1, 13):
    gain = 1.0 if month <= 6 else 1.6            # 7월에 센서 펌웨어가 바뀌어 같은 떨림이 1.6배로 찍힌다고 가정
    d = site(300, 65, -1.4, gain=gain, noise=0.3)
    log.append((month, d.y.sum(), m.predict_proba(d[X])[:, 1].sum()))
log = pd.DataFrame(log, columns=["month", "O", "E"])
log["O/E"] = log.O / log.E
log["O/E_3m"] = log.O.rolling(3).sum() / log.E.rolling(3).sum()
# 0.9는 이 예시에서 정한 규칙이다. 실제 규칙은 표본 크기와 허용 오차를 보고 미리 정한다
log["flag"] = np.where(log["O/E_3m"] < 0.9, "점검", "")
print(log.round(2).to_string(index=False))